# BatikCraft — LoRA Batik untuk Objek Apa Pun

Train LoRA gaya batik dari dataset batik saja. Wayang dan objek lain tidak perlu ada
di dataset. File gambar korup otomatis dilewati dan dicatat ke `skipped-images.json`.
Output: `pytorch_lora_weights.safetensors`, PNG transparan, dan `.batikmodel`.


In [ ]:
import importlib, subprocess, sys
pkgs={'diffusers':'diffusers==0.39.0','transformers':'transformers>=4.48,<5',
'accelerate':'accelerate>=1.2','peft':'peft>=0.17','datasets':'datasets>=3.2',
'safetensors':'safetensors>=0.4','torchvision':'torchvision','cv2':'opencv-python-headless>=4.10'}
missing=[]
for module,package in pkgs.items():
    try: importlib.import_module(module)
    except ImportError: missing.append(package)
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])


In [ ]:
from dataclasses import dataclass,asdict
from pathlib import Path
@dataclass
class CFG:
    dataset_root:str=''
    object_path:str=''
    mask_path:str=''
    motif_reference_path:str=''
    base_model:str='stable-diffusion-v1-5/stable-diffusion-v1-5'
    controlnet_model:str='lllyasviel/control_v11p_sd15_canny'
    output_dir:str='/kaggle/working/batikcraft-any-object-lora'
    trigger_word:str='bcr_batikstyle'
    model_id:str='batikcraft-style-any-object-v1'
    author:str='Balya Rochmadi'
    resolution:int=512
    max_images:int=0
    max_train_steps:int=1600
    batch_size:int=1
    grad_accum:int=4
    learning_rate:float=1e-4
    rank:int=16
    seed:int=2026
    lora_weight:float=.85
    strength:float=.38
    controlnet_scale:float=.90
    outline_strength:float=.65
    use_controlnet:bool=True
    low_vram:bool=False
cfg=CFG();Path(cfg.output_dir).mkdir(parents=True,exist_ok=True);print(asdict(cfg))
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Accelerator Kaggle.')
print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0))


In [ ]:
import json,random,shutil
from pathlib import Path
from PIL import Image,ImageFile,ImageOps,UnidentifiedImageError
ImageFile.LOAD_TRUNCATED_IMAGES=True
EXT={'.png','.jpg','.jpeg','.webp','.bmp','.tif','.tiff'}
TAGS={'parang':'parang diagonal rhythm','kawung':'kawung geometric rosette',
'ceplok':'ceplok repeat','mega':'mega mendung cloud ornament','lereng':'lereng arrangement',
'floral':'floral tendril ornament','ornamen':'ornamental composition'}
def images(root): return sorted(p for p in Path(root).rglob('*') if p.is_file() and p.suffix.lower() in EXT)
def discover():
    choices=[(len(images(p)),p) for p in Path('/kaggle/input').iterdir() if p.is_dir()]
    choices=[x for x in choices if x[0]]
    if not choices: raise RuntimeError('Dataset gambar tidak ditemukan.')
    return max(choices)[1]
def caption(p):
    name='_'.join(x.lower().replace('-','_') for x in p.parts)
    tags=[v for k,v in TAGS.items() if k in name] or ['ornamental Indonesian batik composition']
    return f"{cfg.trigger_word}, authentic Indonesian batik textile style, {', '.join(tags)}, wax-resist canting linework, intricate isen-isen"
def load_image(path):
    with Image.open(path) as probe: probe.verify()
    with Image.open(path) as image:
        image=ImageOps.exif_transpose(image);image.load();return image.convert('RGB')
root=Path(cfg.dataset_root) if cfg.dataset_root else discover()
candidates=images(root);random.Random(cfg.seed).shuffle(candidates)
if cfg.max_images>0:candidates=candidates[:cfg.max_images]
train=Path(cfg.output_dir)/'prepared'/'train'
if train.parent.exists():shutil.rmtree(train.parent)
train.mkdir(parents=True);accepted=[];skipped=[]
with (train/'metadata.jsonl').open('w',encoding='utf-8') as handle:
    for path in candidates:
        try:
            image=load_image(path)
            fitted=ImageOps.fit(image,(cfg.resolution,cfg.resolution),method=Image.Resampling.LANCZOS)
            output=train/f'{len(accepted):06d}.png';fitted.save(output,format='PNG',optimize=True)
        except (UnidentifiedImageError,OSError,ValueError,SyntaxError) as exc:
            skipped.append({'path':str(path),'error':f'{type(exc).__name__}: {exc}'})
            print('SKIP rusak/tidak valid:',path,'->',type(exc).__name__,exc);continue
        handle.write(json.dumps({'file_name':output.name,'text':caption(path.relative_to(root))},ensure_ascii=False)+'\n')
        accepted.append(path)
prepared_count=len(accepted)
report=Path(cfg.output_dir)/'skipped-images.json'
report.write_text(json.dumps({'dataset_root':str(root),'candidate_count':len(candidates),
'accepted_count':prepared_count,'skipped_count':len(skipped),'skipped':skipped},
ensure_ascii=False,indent=2),encoding='utf-8')
if prepared_count<2:raise RuntimeError('Gambar valid kurang dari 2. Periksa skipped-images.json.')
files=accepted
print('dataset',root,'kandidat',len(candidates),'valid',prepared_count,'skip',len(skipped))
print('laporan',report,'train',train)


In [ ]:
import gc,json,math,torch,torch.nn.functional as F
from accelerate import Accelerator
from datasets import load_dataset
from diffusers import AutoencoderKL,DDPMScheduler,StableDiffusionPipeline,UNet2DConditionModel
from diffusers.optimization import get_scheduler
from diffusers.utils import convert_state_dict_to_diffusers
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict
from torch.utils.data import DataLoader
from torchvision import transforms
from transformers import CLIPTextModel,CLIPTokenizer
acc=Accelerator(gradient_accumulation_steps=cfg.grad_accum,mixed_precision='fp16')
tok=CLIPTokenizer.from_pretrained(cfg.base_model,subfolder='tokenizer')
text=CLIPTextModel.from_pretrained(cfg.base_model,subfolder='text_encoder')
vae=AutoencoderKL.from_pretrained(cfg.base_model,subfolder='vae')
unet=UNet2DConditionModel.from_pretrained(cfg.base_model,subfolder='unet')
noise=DDPMScheduler.from_pretrained(cfg.base_model,subfolder='scheduler')
vae.requires_grad_(False);text.requires_grad_(False);unet.requires_grad_(False)
unet.add_adapter(LoraConfig(r=cfg.rank,lora_alpha=cfg.rank,init_lora_weights='gaussian',
target_modules=['to_k','to_q','to_v','to_out.0']));unet.enable_gradient_checkpointing()
ds=load_dataset('imagefolder',data_dir=str(train.parent),split='train')
tf=transforms.Compose([transforms.Resize(cfg.resolution),transforms.CenterCrop(cfg.resolution),
transforms.RandomHorizontalFlip(.1),transforms.ToTensor(),transforms.Normalize([.5],[.5])])
def prep(x):
    x['pixel_values']=[tf(im.convert('RGB')) for im in x['image']]
    x['input_ids']=tok(list(x['text']),max_length=tok.model_max_length,padding='max_length',
    truncation=True,return_tensors='pt').input_ids
    return x
ds=ds.with_transform(prep)
def collate(x):return {'pixel_values':torch.stack([e['pixel_values'] for e in x]).float(),
'input_ids':torch.stack([e['input_ids'] for e in x])}
loader=DataLoader(ds,shuffle=True,collate_fn=collate,batch_size=cfg.batch_size,num_workers=2)
params=[p for p in unet.parameters() if p.requires_grad]
opt=torch.optim.AdamW(params,lr=cfg.learning_rate,weight_decay=1e-2)
updates=math.ceil(len(loader)/cfg.grad_accum);epochs=math.ceil(cfg.max_train_steps/max(updates,1))
lr=get_scheduler('constant_with_warmup',optimizer=opt,num_warmup_steps=min(100,cfg.max_train_steps//10),
num_training_steps=cfg.max_train_steps)
unet,opt,loader,lr=acc.prepare(unet,opt,loader,lr)
vae.to(acc.device,dtype=torch.float16);text.to(acc.device,dtype=torch.float16)
step=0;losses=[]
for _epoch in range(epochs):
    unet.train()
    for batch in loader:
        with acc.accumulate(unet):
            lat=vae.encode(batch['pixel_values'].to(acc.device,dtype=torch.float16)).latent_dist.sample()*vae.config.scaling_factor
            eps=torch.randn_like(lat);t=torch.randint(0,noise.config.num_train_timesteps,(lat.shape[0],),device=lat.device).long()
            pred=unet(noise.add_noise(lat,eps,t),t,text(batch['input_ids'].to(acc.device))[0]).sample
            target=noise.get_velocity(lat,eps,t) if noise.config.prediction_type=='v_prediction' else eps
            loss=F.mse_loss(pred.float(),target.float());acc.backward(loss)
            if acc.sync_gradients:acc.clip_grad_norm_(params,1.0)
            opt.step();lr.step();opt.zero_grad(set_to_none=True)
        if acc.sync_gradients:
            step+=1;losses.append(float(loss.detach()))
            if step%20==0:print(step,cfg.max_train_steps,sum(losses[-20:])/len(losses[-20:]))
        if step>=cfg.max_train_steps:break
    if step>=cfg.max_train_steps:break
acc.wait_for_everyone();lora_dir=Path(cfg.output_dir)/'lora';lora_dir.mkdir(exist_ok=True)
if acc.is_main_process:
    state=convert_state_dict_to_diffusers(get_peft_model_state_dict(acc.unwrap_model(unet)))
    StableDiffusionPipeline.save_lora_weights(str(lora_dir),unet_lora_layers=state,safe_serialization=True)
    (Path(cfg.output_dir)/'training-report.json').write_text(json.dumps({'steps':step,
    'samples':prepared_count,'skipped_images':len(skipped),'mean_loss':sum(losses[-100:])/len(losses[-100:]),
    'object_dataset_required':False},indent=2))
lora_path=lora_dir/'pytorch_lora_weights.safetensors';print(lora_path)
del unet,vae,text,loader;gc.collect();torch.cuda.empty_cache()


In [ ]:
import cv2,hashlib,numpy as np,zipfile
from PIL import Image,ImageChops,ImageFilter,ImageOps
from diffusers import AutoPipelineForImage2Image,ControlNetModel,StableDiffusionControlNetImg2ImgPipeline
def mask_of(src):
    alpha=src.getchannel('A')
    if alpha.getextrema()[0]<255:return alpha
    arr=np.asarray(src.convert('RGB'),dtype=np.int16)
    bg=np.median(np.stack([arr[0,0],arr[0,-1],arr[-1,0],arr[-1,-1]]),axis=0)
    return Image.fromarray((np.sqrt(np.sum((arr-bg)**2,axis=2))>=38).astype(np.uint8)*255).filter(ImageFilter.MaxFilter(3))
def batify_object():
    if not cfg.object_path:raise RuntimeError('Isi cfg.object_path terlebih dahulu.')
    with Image.open(cfg.object_path) as image:src=ImageOps.exif_transpose(image).convert('RGBA')
    if cfg.mask_path:
        with Image.open(cfg.mask_path) as image:mask=image.convert('L').resize(src.size)
    else:mask=mask_of(src)
    box=mask.getbbox()
    if box is None:raise RuntimeError('Mask objek kosong.')
    crop=src.crop(box);cm=mask.crop(box);scale=min(cfg.resolution*.84/crop.width,cfg.resolution*.84/crop.height)
    size=(max(1,round(crop.width*scale)),max(1,round(crop.height*scale)));crop=crop.resize(size);cm=cm.resize(size)
    left=(cfg.resolution-size[0])//2;top=(cfg.resolution-size[1])//2
    init=Image.new('RGB',(cfg.resolution,cfg.resolution),'white');init.paste(crop.convert('RGB'),(left,top),cm)
    sm=Image.new('L',init.size);sm.paste(cm,(left,top))
    edge=np.maximum(cv2.Canny(cv2.cvtColor(np.asarray(init),cv2.COLOR_RGB2GRAY),70,160),
    cv2.Canny(np.asarray(sm),50,120));control=Image.fromarray(edge).convert('RGB')
    if cfg.use_controlnet:
        cn=ControlNetModel.from_pretrained(cfg.controlnet_model,torch_dtype=torch.float16)
        pipe=StableDiffusionControlNetImg2ImgPipeline.from_pretrained(cfg.base_model,controlnet=cn,
        torch_dtype=torch.float16,safety_checker=None,requires_safety_checker=False)
    else:pipe=AutoPipelineForImage2Image.from_pretrained(cfg.base_model,torch_dtype=torch.float16,
    safety_checker=None,requires_safety_checker=False)
    pipe.load_lora_weights(str(lora_path.parent),weight_name=lora_path.name,adapter_name='batik_style')
    pipe.set_adapters(['batik_style'],adapter_weights=[cfg.lora_weight])
    pipe.enable_model_cpu_offload() if cfg.low_vram else pipe.to('cuda')
    kw={'prompt':f'{cfg.trigger_word}, same isolated object transformed into authentic Indonesian batik ornament, preserve exact identity pose proportions and silhouette, wax-resist canting lines, isen-isen',
    'negative_prompt':'different object, changed silhouette, changed pose, extra object, photograph, background scene, text, watermark, blurry',
    'image':init,'strength':cfg.strength,'num_inference_steps':28,'guidance_scale':6.5,
    'generator':torch.Generator('cuda').manual_seed(cfg.seed)}
    if cfg.use_controlnet:kw.update(control_image=control,controlnet_conditioning_scale=cfg.controlnet_scale)
    raw=pipe(**kw).images[0];sb=sm.getbbox();target=(box[2]-box[0],box[3]-box[1])
    result=raw.convert('RGBA').crop(sb).resize(target);tm=mask.crop(box).resize(target);result.putalpha(tm)
    detail=ImageOps.autocontrast(src.crop(box).resize(target).convert('L')).filter(ImageFilter.FIND_EDGES)
    silhouette=ImageChops.subtract(tm.filter(ImageFilter.MaxFilter(5)),tm.filter(ImageFilter.MinFilter(3)))
    outline=ImageChops.multiply(ImageChops.lighter(detail,silhouette),tm).point(lambda v:int(v*cfg.outline_strength))
    ink=Image.new('RGBA',target,(45,25,17,0));ink.putalpha(outline);result.alpha_composite(ink);result.putalpha(tm)
    output=Image.new('RGBA',src.size,(0,0,0,0));output.alpha_composite(result,(box[0],box[1]))
    path=Path(cfg.output_dir)/'batified-object.png';output.save(path);print(path);return output
def build_batikmodel():
    data=lora_path.read_bytes()
    manifest={'format':'batikcraft-model-pack','schema_version':'1.0','model':{'id':cfg.model_id,
    'name':'BatikCraft Any Object LoRA','version':'1.0.0','type':'lora','base_model_family':'sd15',
    'trigger_words':[cfg.trigger_word],'recommended_weight':cfg.lora_weight,'resolution':cfg.resolution,
    'capabilities':['object','selection','inpainting','structured-output'],
    'lora_file':'lora/pytorch_lora_weights.safetensors','author':cfg.author,
    'description':'Style-only Batik LoRA. Object classes are supplied at inference.','license':'',
    'controlnet_family':'canny','negative_prompt':'different object, changed silhouette, text, watermark, blurry',
    'metadata':{'object_dataset_required':False,'seed':cfg.seed,'steps':cfg.max_train_steps,'rank':cfg.rank}},
    'files':[{'path':'lora/pytorch_lora_weights.safetensors','role':'lora',
    'sha256':hashlib.sha256(data).hexdigest(),'size':len(data)}]}
    path=Path(cfg.output_dir)/f'{cfg.model_id}.batikmodel'
    with zipfile.ZipFile(path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as archive:
        archive.writestr('manifest.json',json.dumps(manifest,ensure_ascii=False,indent=2))
        archive.writestr('lora/pytorch_lora_weights.safetensors',data)
    print(path);return path
# result=batify_object()
batikmodel_path=build_batikmodel()
